In [ ]:
# from pathlib import Path
# import json, yaml
# import pandas as pd
# from scripts.data import DataConfig, _prepare_frame, get_dataloaders

# def check_prepare(profile_path):
#     cfg = yaml.safe_load(Path(profile_path).read_text())
#     dc = DataConfig(**cfg["data"], dataset=cfg["dataset"])
#     df = _prepare_frame(dc)
#     print(f"\n=== prepare {profile_path} ===")
#     print("dataset key:", dc.dataset)
#     print("target_col present?", dc.target_col in df.columns)
#     print("market_value in df?", "market_value" in df.columns)
#     print("dropped_cols recorded:", df.attrs.get("dropped_cols", []))
#     print("col count after drops:", len(df.columns))

# def check_meta(profile_path, sample_rows=2000):
#     cfg = yaml.safe_load(Path(profile_path).read_text())
#     dc = DataConfig(**cfg["data"], dataset=cfg["dataset"])
#     src = Path(dc.csv_path)
#     tmp = src.with_name(src.stem + "_sample.csv")
#     pd.read_csv(src, nrows=sample_rows).to_csv(tmp, index=False)

#     train_cfg = {
#         "data": {**dc.__dict__, "csv_path": str(tmp)},
#         "model": {"head_type": "point", "hidden_dims": [8], "activation": "relu"},
#         "training": {"lr": 1e-3, "epochs": 1, "device": "cpu", "point_loss": "mse"},
#         "objective": "val_mae",
#         "io": {"outdir": f"outputs/tmp_verify_{dc.dataset}"},
#     }
#     get_dataloaders(train_cfg, seed=0)
#     meta = json.loads((Path(train_cfg["io"]["outdir"]) / "preproc_meta.json").read_text())
#     print(f"\n=== meta {profile_path} ===")
#     print("dataset:", meta.get("dataset"))
#     print("target_col:", meta.get("target_col"))
#     print("csv_path:", meta.get("csv_path"))
#     print("dropped_cols:", meta.get("dropped_cols"))

# check_prepare("configs/datasets/craigslist_data.yaml")
# check_prepare("configs/datasets/vw_data.yaml")
# check_meta("configs/datasets/craigslist_data.yaml")
# check_meta("configs/datasets/vw_data.yaml")



=== prepare configs/datasets/craigslist_data.yaml ===
dataset key: craigs
target_col present? True
market_value in df? False
dropped_cols recorded: []
col count after drops: 26

=== prepare configs/datasets/vw_data.yaml ===
dataset key: vw
target_col present? True
market_value in df? False
dropped_cols recorded: ['Unnamed: 0', 'market_value']
col count after drops: 13

=== meta configs/datasets/craigslist_data.yaml ===
dataset: craigs
target_col: price
csv_path: /home/canelo/uncertainty_quantification/datasets/craigslist_cleaned_sample.csv
dropped_cols: []

=== meta configs/datasets/vw_data.yaml ===
dataset: vw
target_col: y_true
csv_path: /home/canelo/uncertainty_quantification/datasets/rv_dataset_2024_sample.csv
dropped_cols: ['Unnamed: 0', 'market_value']


In [6]:
import sys, importlib, json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("/home/canelo/uncertainty_quantification")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.config_resolver as config_resolver
importlib.reload(config_resolver)
from scripts import train_regression
importlib.reload(train_regression)
from scripts.data import get_dataloaders

def resolve_config(cfg_path, dataset=None):
    cfg = config_resolver.load_and_resolve_config(cfg_path, dataset_override=dataset)
    print(f"\n=== {cfg_path} (dataset={dataset or cfg.get('dataset')}) ===")
    print("dataset key:", cfg.get("dataset") or cfg.get("data", {}).get("dataset"))
    print("csv_path:", cfg.get("data", {}).get("csv_path"))
    print("target_col:", cfg.get("data", {}).get("target_col"))
    io = cfg.get("io", {}) or {}
    print("io.outdir:", io.get("outdir"))
    print("io.training_outdir:", io.get("training_outdir"))
    print("io.evals_root:", io.get("evals_root"))
    return cfg

def smoke_run(cfg_path, dataset_key, sample_rows=1000):
    cfg = config_resolver.load_and_resolve_config(cfg_path, dataset_override=dataset_key)
    src = Path(cfg["data"]["csv_path"])
    tmp = src.with_name(src.stem + f"_sample_{dataset_key}.csv")
    pd.read_csv(src, nrows=sample_rows).to_csv(tmp, index=False)
    cfg["data"]["csv_path"] = str(tmp)
    cfg.setdefault("io", {})
    cfg["io"]["outdir"] = f"outputs/tmp_phase3_{dataset_key}"
    cfg.setdefault("model", {"head_type": "point", "hidden_dims": [8], "activation": "relu"})
    cfg.setdefault("training", {"lr": 1e-3, "epochs": 1, "device": "cpu", "point_loss": "mse"})
    cfg.setdefault("objective", "val_mae")
    get_dataloaders(cfg, seed=0)
    meta = json.loads((Path(cfg["io"]["outdir"]) / "preproc_meta.json").read_text())
    print(f"\nmeta for {dataset_key}: dataset={meta.get('dataset')} target_col={meta.get('target_col')} dropped_cols={meta.get('dropped_cols')}")

resolve_config("configs/train_base.yaml", dataset="craigs")
resolve_config("configs/train_base.yaml", dataset="vw")
smoke_run("configs/train_base.yaml", "craigs", sample_rows=1000)
smoke_run("configs/train_base.yaml", "vw", sample_rows=1000)



=== configs/train_base.yaml (dataset=craigs) ===
dataset key: craigs
csv_path: datasets/craigslist_cleaned.csv
target_col: price
io.outdir: craigs_
io.training_outdir: outputs/trainings/base/craigs_{job}_{ts}_{jobid}
io.evals_root: outputs/craigs_evals

=== configs/train_base.yaml (dataset=vw) ===
dataset key: vw
csv_path: datasets/rv_dataset_2024.csv
target_col: y_true
io.outdir: vw_
io.training_outdir: outputs/trainings/base/vw_{job}_{ts}_{jobid}
io.evals_root: outputs/vw_evals

meta for craigs: dataset=craigs target_col=price dropped_cols=[]

meta for vw: dataset=vw target_col=y_true dropped_cols=['Unnamed: 0', 'market_value']


In [7]:
# Phase 4 validation: inspect the kNN OOD configs and build the commands to run them
from pathlib import Path
import yaml

def show_knn_cfg(path):
    cfg = yaml.safe_load(Path(path).read_text())
    k = cfg.get("knn_ood", {})
    print(f"\n=== {path} ===")
    print("csv_path:", k.get("csv_path"))
    print("preproc_meta:", k.get("preproc_meta"))
    print("out_dir:", k.get("out_dir"))
    print("out_prefix:", k.get("out_prefix"))
    print("id_q:", k.get("id_q"), "ood_q:", k.get("ood_q"))
    print("knn_k_list:", k.get("knn_k_list"))
    print("proj_dim:", k.get("proj_dim"))
    print("chunk_rows:", k.get("chunk_rows"))

show_knn_cfg("configs/knn_ood_craigs.yaml")
show_knn_cfg("configs/knn_ood_vw.yaml")

# Build the run commands (replace preproc_meta with the actual base run)
cmd_craigs = "python scripts/make_knn_ood_knn.py --config configs/knn_ood_craigs.yaml --run-mode local"
cmd_vw = "python scripts/make_knn_ood_knn.py --config configs/knn_ood_vw.yaml --run-mode local"
print("\nRun commands (update preproc_meta before executing):")
print(cmd_craigs)
print(cmd_vw)



=== configs/knn_ood_craigs.yaml ===
csv_path: datasets/craigslist_cleaned_FULL.csv
preproc_meta: outputs/trainings/base/craigs_run/preproc_meta.json
out_dir: datasets/knn_ood
out_prefix: craigs
id_q: [0.0, 0.8] ood_q: [0.95, 1.0]
knn_k_list: [3, 5, 8, 12]
proj_dim: 32
chunk_rows: 20000

=== configs/knn_ood_vw.yaml ===
csv_path: datasets/rv_dataset_2024.csv
preproc_meta: outputs/trainings/base/vw_run/preproc_meta.json
out_dir: datasets/knn_ood
out_prefix: vw
id_q: [0.0, 0.8] ood_q: [0.95, 1.0]
knn_k_list: [3, 5, 8, 12]
proj_dim: 32
chunk_rows: 20000

Run commands (update preproc_meta before executing):
python scripts/make_knn_ood_knn.py --config configs/knn_ood_craigs.yaml --run-mode local
python scripts/make_knn_ood_knn.py --config configs/knn_ood_vw.yaml --run-mode local


In [8]:
# Phase 4 + 5 sanity: inspect kNN OOD configs and build commands, and dry-run OOD eval configs

from pathlib import Path
import yaml

def show_knn(path):
    cfg = yaml.safe_load(Path(path).read_text())
    k = cfg.get("knn_ood", {})
    print(f"\n=== {path} ===")
    print("csv_path:", k.get("csv_path"))
    print("preproc_meta:", k.get("preproc_meta"))
    print("out_dir:", k.get("out_dir"))
    print("out_prefix:", k.get("out_prefix"))
    print("id_q:", k.get("id_q"), "ood_q:", k.get("ood_q"))
    print("knn_k_list:", k.get("knn_k_list"))

def show_ood_eval(path):
    cfg = yaml.safe_load(Path(path).read_text()) or {}
    oods = cfg.get("ood_sets", {}) or {}
    print(f"\n=== {path} ===")
    print("run_tag:", cfg.get("run_tag"))
    print("enabled ood sets:", {k:v.get("csv") for k,v in oods.items() if v.get("enabled")})
    print("methods regression enabled:", cfg.get("methods", {}).get("regression", {}).get("enabled", False))
    print("roots:", cfg.get("methods", {}).get("regression", {}).get("roots"))

show_knn("configs/knn_ood_craigs.yaml")
show_knn("configs/knn_ood_vw.yaml")

show_ood_eval("configs/ood_eval_all_methods_craigs.yaml")
show_ood_eval("configs/ood_eval_all_methods_vw.yaml")

print("\nRun these after updating preproc_meta in the kNN configs to real paths:")
print("python scripts/make_knn_ood_knn.py --config configs/knn_ood_craigs.yaml --run-mode local")
print("python scripts/make_knn_ood_knn.py --config configs/knn_ood_vw.yaml --run-mode local")

print("\nDry-run OOD evals (after kNN CSVs exist):")
print("python scripts/run_ood_eval_all_methods.py --config configs/ood_eval_all_methods_craigs.yaml --run-tag \"_l_\" --dry-run")
print("python scripts/run_ood_eval_all_methods.py --config configs/ood_eval_all_methods_vw.yaml --run-tag \"_l_\" --dry-run")



=== configs/knn_ood_craigs.yaml ===
csv_path: datasets/craigslist_cleaned_FULL.csv
preproc_meta: outputs/trainings/base/craigs_run/preproc_meta.json
out_dir: datasets/knn_ood
out_prefix: craigs
id_q: [0.0, 0.8] ood_q: [0.95, 1.0]
knn_k_list: [3, 5, 8, 12]

=== configs/knn_ood_vw.yaml ===
csv_path: datasets/rv_dataset_2024.csv
preproc_meta: outputs/trainings/base/vw_run/preproc_meta.json
out_dir: datasets/knn_ood
out_prefix: vw
id_q: [0.0, 0.8] ood_q: [0.95, 1.0]
knn_k_list: [3, 5, 8, 12]

=== configs/ood_eval_all_methods_craigs.yaml ===
run_tag: _l_
enabled ood sets: {'knn_hard': 'datasets/knn_ood/craigs_ood_knn_hard.csv', 'knn_mild': 'datasets/knn_ood/craigs_ood_knn_mild.csv'}
methods regression enabled: True
roots: ['outputs/trainings', 'outputs/trainings/ensembles']

=== configs/ood_eval_all_methods_vw.yaml ===
run_tag: _l_
enabled ood sets: {'knn_hard': 'datasets/knn_ood/vw_ood_knn_hard.csv', 'knn_mild': 'datasets/knn_ood/vw_ood_knn_mild.csv'}
methods regression enabled: True
root